# LangGraph Crash Course

A hands-on introduction to building stateful, graph-based AI agents with LangGraph.

---

## Table of Contents

1. What is LangGraph?
2. Installation and Setup
3. Core Concepts
4. Your First Graph
5. State Management
6. Conditional Edges and Branching
7. Tool Use and the ReAct Pattern
8. Human-in-the-Loop
9. Real-World Example: Research and Summarization Agent
10. Advanced: Multi-Agent Graphs and Subgraphs
11. Best Practices and Tips

---
## 1. What is LangGraph?

LangGraph is a library from LangChain for building stateful, multi-actor applications with LLMs. Unlike linear chains or simple agent loops, LangGraph models your application as a **directed graph** where:

- **Nodes** are Python functions or LLM calls that do work
- **Edges** define the flow between nodes, which can be conditional
- **State** is a shared data structure that every node can read from and write to

This makes it possible to build agents that loop, branch, recover from errors, pause for human input, and coordinate multiple sub-agents — all in a transparent, inspectable way.

### How LangGraph Differs from CrewAI

| Dimension | LangGraph | CrewAI |
|-----------|-----------|--------|
| Abstraction level | Low — you define nodes and edges explicitly | High — roles, goals, backstories |
| Control flow | Graph (arbitrary routing) | Sequential or hierarchical |
| State management | Explicit, typed state schema | Implicit via task context |
| Best for | Complex loops, branching, fine-grained control | Quick multi-agent pipelines |

LangGraph gives you more control at the cost of more explicit wiring.

---
## 2. Installation and Setup

In [ ]:
!pip install langgraph langchain langchain-openai langchain-anthropic

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"

# To use Anthropic Claude:
# os.environ["ANTHROPIC_API_KEY"] = "your-anthropic-api-key-here"

print("Environment configured.")

---
## 3. Core Concepts

### The Three Pillars of LangGraph

| Concept | Description |
|---------|-------------|
| **State** | A typed dictionary (via `TypedDict` or Pydantic) shared across all nodes. Every node receives the current state and returns an update to it. |
| **Node** | A Python function that takes state as input and returns a partial state update. This is where logic and LLM calls live. |
| **Edge** | A directed connection between nodes. Edges can be unconditional (`add_edge`) or conditional (`add_conditional_edges`), where a function decides which node to visit next. |

### Graph Lifecycle

1. Define the state schema
2. Create a `StateGraph` with that schema
3. Add nodes (functions)
4. Add edges between nodes
5. Set an entry point
6. Compile the graph
7. Invoke or stream the compiled graph with an initial state

### Special Nodes

- `START` — a virtual node representing the graph's entry point
- `END` — a virtual node that terminates the graph

---
## 4. Your First Graph

A minimal two-node graph: one node generates a joke, the next evaluates it.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# Step 1: Define the state
class JokeState(TypedDict):
    topic: str
    joke: str
    evaluation: str

# Step 2: Define nodes
def generate_joke(state: JokeState) -> dict:
    """Generate a joke about the given topic."""
    response = llm.invoke(f"Tell me a short, clever joke about: {state['topic']}")
    return {"joke": response.content}

def evaluate_joke(state: JokeState) -> dict:
    """Rate the joke on a scale of 1-10."""
    response = llm.invoke(
        f"Rate this joke from 1 to 10 and explain why:\n\n{state['joke']}"
    )
    return {"evaluation": response.content}

# Step 3: Build the graph
graph = StateGraph(JokeState)

graph.add_node("generate", generate_joke)
graph.add_node("evaluate", evaluate_joke)

graph.add_edge(START, "generate")
graph.add_edge("generate", "evaluate")
graph.add_edge("evaluate", END)

# Step 4: Compile and run
app = graph.compile()

result = app.invoke({"topic": "quantum physics", "joke": "", "evaluation": ""})
print("Joke:", result["joke"])
print("\nEvaluation:", result["evaluation"])

In [ ]:
# Visualize the graph structure (requires graphviz or Mermaid support)
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    # Fall back to Mermaid text if PNG rendering is unavailable
    print(app.get_graph().draw_mermaid())

---
## 5. State Management

State is the backbone of LangGraph. Every node receives the full current state and returns a dictionary with only the keys it wants to update.

### Reducers: Controlling How State Is Updated

By default, a node's return value **replaces** the existing value for each key. You can use **reducers** to change this behavior — for example, to append messages to a list instead of overwriting them.

In [ ]:
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage

# add_messages is a built-in reducer that appends to the messages list
# instead of replacing it
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]  # Annotated with reducer
    turn_count: int

def chat_node(state: ChatState) -> dict:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini")
    response = llm.invoke(state["messages"])
    return {
        "messages": [response],      # add_messages appends this
        "turn_count": state["turn_count"] + 1,
    }

graph = StateGraph(ChatState)
graph.add_node("chat", chat_node)
graph.add_edge(START, "chat")
graph.add_edge("chat", END)
app = graph.compile()

result = app.invoke({
    "messages": [HumanMessage(content="What is a graph database?")],
    "turn_count": 0,
})

for msg in result["messages"]:
    role = "Human" if isinstance(msg, HumanMessage) else "AI"
    print(f"{role}: {msg.content}\n")

### Using Pydantic for State

You can use a Pydantic `BaseModel` instead of `TypedDict` for validation and default values.

In [ ]:
from pydantic import BaseModel, Field
from typing import Annotated, List
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage

class AgentState(BaseModel):
    messages: Annotated[List[BaseMessage], add_messages] = Field(default_factory=list)
    iterations: int = 0
    final_answer: str = ""
    error: str = ""

# Use AgentState exactly as you would a TypedDict in StateGraph(AgentState)

---
## 6. Conditional Edges and Branching

Conditional edges allow the graph to make routing decisions at runtime based on the current state. This is how you implement loops, retries, and branching logic.

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
import json

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

class ReviewState(TypedDict):
    draft: str
    feedback: str
    revision_count: int
    approved: bool

def write_draft(state: ReviewState) -> dict:
    if not state["draft"]:
        response = llm.invoke("Write a 3-sentence summary of the water cycle.")
        return {"draft": response.content, "revision_count": 0}
    # Revise based on feedback
    response = llm.invoke(
        f"Revise this text based on the feedback below.\n"
        f"Text: {state['draft']}\nFeedback: {state['feedback']}"
    )
    return {"draft": response.content, "revision_count": state["revision_count"] + 1}

def review_draft(state: ReviewState) -> dict:
    response = llm.invoke(
        f"Review this text. Reply with JSON: {{\"approved\": true/false, \"feedback\": \"...\"}}.\n"
        f"Text: {state['draft']}"
    )
    data = json.loads(response.content.strip("```json\n").strip("```"))
    return {"approved": data["approved"], "feedback": data.get("feedback", "")}

# Routing function — returns the name of the next node
def route_after_review(state: ReviewState) -> Literal["write", "__end__"]:
    if state["approved"] or state["revision_count"] >= 3:
        return "__end__"  # Stop the graph
    return "write"        # Loop back and revise

graph = StateGraph(ReviewState)
graph.add_node("write", write_draft)
graph.add_node("review", review_draft)

graph.add_edge(START, "write")
graph.add_edge("write", "review")

# Conditional edge: after review, call route_after_review to decide
graph.add_conditional_edges("review", route_after_review)

app = graph.compile()

result = app.invoke({"draft": "", "feedback": "", "revision_count": 0, "approved": False})
print("Final draft:", result["draft"])
print("Revisions made:", result["revision_count"])

---
## 7. Tool Use and the ReAct Pattern

The **ReAct** pattern (Reason + Act) is the standard way to build tool-using agents. The agent reasons about what to do, calls a tool, observes the result, and repeats until it has a final answer.

LangGraph ships with a prebuilt `create_react_agent` for this pattern.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# Define tools using the @tool decorator
@tool
def get_word_count(text: str) -> str:
    """Count the number of words in the given text."""
    return str(len(text.split()))

@tool
def reverse_string(text: str) -> str:
    """Reverse the characters in a string."""
    return text[::-1]

@tool
def calculate(expression: str) -> str:
    """Evaluate a simple math expression like '2 + 2' or '10 * 5'."""
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"

llm = ChatOpenAI(model="gpt-4o", temperature=0)
tools = [get_word_count, reverse_string, calculate]

# create_react_agent builds the full ReAct graph for you
agent = create_react_agent(llm, tools)

result = agent.invoke({
    "messages": [{"role": "user", "content": "How many words are in 'the quick brown fox', and what is 7 * 8?"}]
})

print(result["messages"][-1].content)

In [ ]:
# Building the ReAct pattern manually gives you full control
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage
from typing import TypedDict

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

llm_with_tools = ChatOpenAI(model="gpt-4o", temperature=0).bind_tools(tools)

def agent_node(state: AgentState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

graph = StateGraph(AgentState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(tools))  # Executes tool calls automatically

graph.add_edge(START, "agent")

# tools_condition: routes to "tools" if the agent made a tool call, else ends
graph.add_conditional_edges("agent", tools_condition)
graph.add_edge("tools", "agent")  # After tools, go back to agent

app = graph.compile()

result = app.invoke({
    "messages": [{"role": "user", "content": "What is 123 * 456?"}]
})
print(result["messages"][-1].content)

---
## 8. Human-in-the-Loop

LangGraph supports pausing graph execution to wait for human input or approval. This is done using **breakpoints** and a **checkpointer**, which saves the graph state between steps so it can be resumed.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage

class ReviewState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def draft_response(state: ReviewState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

def send_response(state: ReviewState) -> dict:
    print("[Sending approved response]:", state["messages"][-1].content)
    return {}

graph = StateGraph(ReviewState)
graph.add_node("draft", draft_response)
graph.add_node("send", send_response)
graph.add_edge(START, "draft")
graph.add_edge("draft", "send")
graph.add_edge("send", END)

# MemorySaver stores checkpoints in memory (use SqliteSaver or PostgresSaver in production)
checkpointer = MemorySaver()

# interrupt_before="send" pauses the graph BEFORE the "send" node runs
app = graph.compile(checkpointer=checkpointer, interrupt_before=["send"])

# Thread config identifies this particular run
config = {"configurable": {"thread_id": "review-session-1"}}

# Run until the breakpoint
state = app.invoke(
    {"messages": [HumanMessage(content="Summarize the benefits of exercise.")]},
    config=config,
)
print("Draft for review:", state["messages"][-1].content)

# --- Human reviews here ---
# To approve and continue:
# app.invoke(None, config=config)

# To provide new input before continuing:
# app.update_state(config, {"messages": [HumanMessage(content="Make it shorter.")]})
# app.invoke(None, config=config)

### Checkpointer Options

| Checkpointer | Use Case |
|--------------|----------|
| `MemorySaver` | Development and testing |
| `SqliteSaver` | Single-process production apps |
| `PostgresSaver` | Multi-process or distributed systems |
| `RedisSaver` | High-throughput, low-latency persistence |

---
## 9. Real-World Example: Research and Summarization Agent

This example builds an agent that searches the web, evaluates whether it has enough information, and iterates until it can produce a high-quality summary.

In [ ]:
from typing import TypedDict, Annotated, List, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
import json

llm = ChatOpenAI(model="gpt-4o", temperature=0)

# State
class ResearchState(TypedDict):
    topic: str
    search_results: List[str]
    summary: str
    quality_score: int      # 1-10; below threshold triggers another search
    iterations: int
    messages: Annotated[List[BaseMessage], add_messages]

# Simulated search tool (replace with a real API like Serper or Tavily)
@tool
def web_search(query: str) -> str:
    """
    Search the web for information about the given query.
    Returns a string with key findings.
    """
    # In a real app, call Tavily, Serper, or another search API here
    return f"[Simulated results for '{query}'] Key points: AI is transforming industries, recent breakthroughs include GPT-4 and Gemini, concerns include safety and job displacement."

# Nodes
def plan_search(state: ResearchState) -> dict:
    """Decide what to search for."""
    response = llm.invoke(
        f"You are researching: {state['topic']}. "
        f"Iteration {state['iterations'] + 1}. "
        f"Previous results: {state['search_results']}. "
        f"Generate the best search query to fill gaps in your knowledge. "
        f"Respond with ONLY the search query, nothing else."
    )
    query = response.content.strip()
    print(f"Searching for: {query}")
    results = web_search.invoke({"query": query})
    return {
        "search_results": state["search_results"] + [results],
        "iterations": state["iterations"] + 1,
    }

def write_summary(state: ResearchState) -> dict:
    """Write a summary from all collected research."""
    all_results = "\n".join(state["search_results"])
    response = llm.invoke(
        f"Write a comprehensive, well-structured summary about '{state['topic']}' "
        f"based on the following research:\n\n{all_results}"
    )
    return {"summary": response.content}

def evaluate_summary(state: ResearchState) -> dict:
    """Score the summary quality from 1-10."""
    response = llm.invoke(
        f"Rate this summary from 1-10 based on depth, accuracy, and completeness. "
        f"Topic: {state['topic']}\nSummary: {state['summary']}\n"
        f"Respond with JSON only: {{\"score\": <number>, \"reason\": \"...\"}}"
    )
    data = json.loads(response.content.strip("```json\n").strip("```"))
    print(f"Quality score: {data['score']}/10 — {data['reason']}")
    return {"quality_score": data["score"]}

# Routing
def should_continue_research(state: ResearchState) -> Literal["search", "__end__"]:
    if state["quality_score"] >= 7 or state["iterations"] >= 3:
        return "__end__"
    return "search"

# Build graph
graph = StateGraph(ResearchState)
graph.add_node("search", plan_search)
graph.add_node("summarize", write_summary)
graph.add_node("evaluate", evaluate_summary)

graph.add_edge(START, "search")
graph.add_edge("search", "summarize")
graph.add_edge("summarize", "evaluate")
graph.add_conditional_edges("evaluate", should_continue_research)

app = graph.compile()

result = app.invoke({
    "topic": "The current state of AI regulation worldwide",
    "search_results": [],
    "summary": "",
    "quality_score": 0,
    "iterations": 0,
    "messages": [],
})

print("\n=== FINAL SUMMARY ===")
print(result["summary"])
print(f"\nCompleted in {result['iterations']} iteration(s) with quality score {result['quality_score']}/10")

---
## 10. Advanced: Multi-Agent Graphs and Subgraphs

### 10.1 Streaming

LangGraph supports streaming at the node level and the token level, which is essential for responsive UIs.

In [ ]:
# Stream node-by-node updates
for chunk in app.stream(
    {
        "topic": "Renewable energy trends",
        "search_results": [],
        "summary": "",
        "quality_score": 0,
        "iterations": 0,
        "messages": [],
    },
    stream_mode="updates",  # emit state updates after each node
):
    node_name = list(chunk.keys())[0]
    print(f"[Node: {node_name}] State updated.")

### 10.2 Subgraphs

A compiled graph can be used as a node inside another graph, enabling modular, composable agent architectures.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# Inner subgraph
class SubState(TypedDict):
    text: str
    processed: str

def process_text(state: SubState) -> dict:
    return {"processed": state["text"].upper()}

subgraph_builder = StateGraph(SubState)
subgraph_builder.add_node("process", process_text)
subgraph_builder.add_edge(START, "process")
subgraph_builder.add_edge("process", END)
subgraph = subgraph_builder.compile()  # Compiled subgraph

# Outer graph
class OuterState(TypedDict):
    text: str
    processed: str
    final_output: str

def finalize(state: OuterState) -> dict:
    return {"final_output": f"Done: {state['processed']}"}

outer_builder = StateGraph(OuterState)
outer_builder.add_node("sub", subgraph)  # Subgraph used as a node
outer_builder.add_node("finalize", finalize)
outer_builder.add_edge(START, "sub")
outer_builder.add_edge("sub", "finalize")
outer_builder.add_edge("finalize", END)
outer_app = outer_builder.compile()

result = outer_app.invoke({"text": "hello world", "processed": "", "final_output": ""})
print(result["final_output"])

### 10.3 Multi-Agent Handoffs

You can model multiple specialized agents as nodes in a shared graph and route between them based on state. This is LangGraph's equivalent of CrewAI's multi-agent crew.

In [ ]:
from typing import TypedDict, Literal, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o", temperature=0)

class MultiAgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    next_agent: str

def make_agent_node(system_prompt: str):
    """Factory that creates a specialized agent node."""
    def node(state: MultiAgentState) -> dict:
        messages = [SystemMessage(content=system_prompt)] + state["messages"]
        response = llm.invoke(messages)
        return {"messages": [response]}
    return node

researcher_node = make_agent_node("You are a research specialist. Gather and present key facts on the topic.")
writer_node = make_agent_node("You are a writer. Take the research and turn it into a polished article.")
editor_node = make_agent_node("You are an editor. Review the article for clarity and suggest final improvements.")

def supervisor(state: MultiAgentState) -> Literal["researcher", "writer", "editor", "__end__"]:
    """Determine which agent should act next based on the conversation history."""
    message_count = len(state["messages"])
    if message_count == 1:
        return "researcher"
    elif message_count == 2:
        return "writer"
    elif message_count == 3:
        return "editor"
    return "__end__"

graph = StateGraph(MultiAgentState)
graph.add_node("researcher", researcher_node)
graph.add_node("writer", writer_node)
graph.add_node("editor", editor_node)

graph.add_conditional_edges(START, supervisor)
graph.add_conditional_edges("researcher", supervisor)
graph.add_conditional_edges("writer", supervisor)
graph.add_conditional_edges("editor", supervisor)

app = graph.compile()

from langchain_core.messages import HumanMessage
result = app.invoke({
    "messages": [HumanMessage(content="Write a short article about the history of the internet.")],
    "next_agent": "",
})

print("Final output:", result["messages"][-1].content)

---
## 11. Best Practices and Tips

### State Design

Keep state fields explicit and typed. Avoid using a single large unstructured dictionary. If state grows complex, consider splitting into a Pydantic model with clear field descriptions. Use reducers (`Annotated` fields) for any field that should accumulate values rather than be overwritten.

### Node Design

Keep nodes small and focused — each node should have one clear responsibility. Nodes that do too much are hard to test and debug. Name nodes as verbs describing what they do: `plan_search`, `write_draft`, `evaluate_quality`.

### Routing Functions

Always add a maximum iteration guard in any loop to prevent infinite cycles. Return explicit `Literal` types from routing functions so LangGraph can validate your graph at compile time.

### Debugging

Use `app.stream(..., stream_mode="updates")` to observe state changes node by node. Visualize the graph with `app.get_graph().draw_mermaid()` to catch wiring mistakes early. Use `MemorySaver` with a fixed `thread_id` during development to replay and inspect execution state.

### Production Considerations

Replace `MemorySaver` with `SqliteSaver` or `PostgresSaver` for persistence across restarts. Use LangGraph's `interrupt_before` or `interrupt_after` for any step that requires human review. Deploy with **LangGraph Platform** (formerly LangGraph Cloud) for hosted execution, monitoring, and a visual debugging studio.

In [ ]:
cheatsheet = """
LangGraph Quick Reference
=========================

from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition, create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# State
class MyState(TypedDict):
    messages: Annotated[list, add_messages]   # Accumulates messages
    counter: int                               # Replaced each update

# Node
def my_node(state: MyState) -> dict:
    return {"counter": state["counter"] + 1}

# Routing function
def my_router(state: MyState) -> Literal["node_a", "node_b", "__end__"]:
    if state["counter"] < 3:
        return "node_a"
    return "__end__"

# Build
graph = StateGraph(MyState)
graph.add_node("my_node", my_node)
graph.add_edge(START, "my_node")
graph.add_edge("my_node", END)                     # Unconditional edge
graph.add_conditional_edges("my_node", my_router)  # Conditional edge

# Compile
checkpointer = MemorySaver()
app = graph.compile(
    checkpointer=checkpointer,
    interrupt_before=["my_node"],  # Pause before this node
)

# Run
config = {"configurable": {"thread_id": "session-1"}}
result = app.invoke({"messages": [], "counter": 0}, config=config)

# Stream
for chunk in app.stream(initial_state, stream_mode="updates"):
    print(chunk)

# Resume after interrupt
app.invoke(None, config=config)

# Prebuilt ReAct agent
llm = ChatOpenAI(model="gpt-4o")
agent = create_react_agent(llm, tools=[my_tool])
agent.invoke({"messages": [{"role": "user", "content": "..."}]})

# Visualize
print(app.get_graph().draw_mermaid())
"""

print(cheatsheet)